# Tier 1 merged analysis (four merged conditions)

Reads `eval_results/tier1_acl_results.csv` and merges `hp`, `hp+sg`, `hp_wiener` into
one `high_pass` condition, analogous to the Tier 2 merged analysis. Outputs go to
`eval_results/derived/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

BASE = Path(r"C:\Users\Leon\Documents\Github\finalproject_cpe027\eval_results")
DERIVED = BASE / "derived"
DERIVED.mkdir(parents=True, exist_ok=True)
RAW_PATH = BASE / "tier1_acl_results.csv"

CONDITIONS = ["arm_a", "high_pass", "hp_ffmpeg", "hp_wiener_oa"]
NONBASE = ["high_pass", "hp_ffmpeg", "hp_wiener_oa"]
KEYS = ["talk_id", "noise_type", "snr_db", "noise_seed"]
B = 5000
SEED = 0

In [ ]:
# Report expected columns not found in the raw CSV.
raw_expected = ["tier","talk_id","condition","noise_type","snr_db","noise_seed","wer","cer",
                "rtf","seconds","audio_duration_s","kter","term_presence_recall",
                "signal_snr_improvement_db","language","duration_bucket","nshr"]
raw = pd.read_csv(RAW_PATH)
print("Raw columns:", raw.columns.tolist())
print("Raw missing expected:", [c for c in raw_expected if c not in raw.columns.tolist()])

In [ ]:
# Prepare data (mirror tier1_analysis.ipynb conventions)
df = raw.copy()
df["snr_db"] = df["snr_db"].fillna("clean").map(lambda v: "clean" if v == "clean" else str(int(float(v))))
df["noise_seed"] = df["noise_seed"].fillna("clean").astype(str)
df["condition"] = df["condition"].astype(str)

# Build four merged conditions: average hp / hp+sg / hp_wiener into high_pass.
high = ["hp", "hp+sg", "hp_wiener"]
hp = df[df.condition.isin(high)].groupby(KEYS, dropna=False)[["wer","cer","rtf","seconds","audio_duration_s"]].mean().reset_index()
hp["condition"] = "high_pass"
other = df[~df.condition.isin(high)].copy()
merged = pd.concat([other, hp], ignore_index=True)
merged["wer_pct"] = 100 * merged["wer"]
merged["cer_pct"] = 100 * merged["cer"]
merged.to_csv(DERIVED / "tier1_merged_conditions.csv", index=False)
print("merged rows:", len(merged), "scenarios:", len(merged) // 4, "talks:", merged.talk_id.nunique())
print("conditions:", merged.condition.value_counts().to_dict())

## 1. Accuracy table (mean WER/CER %, with n)

In [ ]:
acc_rows = []
def acc_for(df_in, scope):
    for c in CONDITIONS:
        g = df_in[df_in.condition == c]
        acc_rows.append({"scope": scope, "condition": c, "n": len(g),
                        "wer_mean_pct": round(100*g["wer"].mean(),4),
                        "cer_mean_pct": round(100*g["cer"].mean(),4)})

acc_for(merged, "overall")
for s in sorted(merged.snr_db.unique()):
    acc_for(merged[merged.snr_db == s], str(s))
acc_df = pd.DataFrame(acc_rows)
print(acc_df.to_string(index=False))
acc_df.to_csv(DERIVED / "tier1_merged_accuracy.csv", index=False)

## 2. Paired deltas, cluster bootstrap over talk_id, Wilcoxon/Holm, Cohen's dz, win/loss/tie

In [ ]:
base = merged[merged.condition == "arm_a"].set_index(KEYS)
delta_rows = []
for c in NONBASE:
    cdf = merged[merged.condition == c].set_index(KEYS)
    common = base.index.intersection(cdf.index)
    for key in common:
        delta_rows.append({
            "talk_id": key[0], "noise_type": key[1], "snr_db": key[2], "noise_seed": key[3],
            "condition": c,
            "delta_wer_pp": 100*(base.loc[key,"wer"] - cdf.loc[key,"wer"]),
            "delta_cer_pp": 100*(base.loc[key,"cer"] - cdf.loc[key,"cer"]),
        })
delta_df = pd.DataFrame(delta_rows)
print("paired deltas:", delta_df.shape)
delta_df.to_csv(DERIVED / "tier1_merged_paired_deltas.csv", index=False)

def cluster_ci(values_df, value_col, cluster_col="talk_id", B=B, seed=SEED):
    clusters = values_df[cluster_col].unique()
    vals = values_df[value_col].to_numpy(float)
    cl = values_df[cluster_col].to_numpy()
    idx_by_cluster = {c: np.where(cl == c)[0] for c in clusters}
    rng = np.random.default_rng(seed)
    boot = np.empty(B)
    for b in range(B):
        chosen = rng.choice(clusters, size=len(clusters), replace=True)
        rows = np.concatenate([idx_by_cluster[c] for c in chosen])
        boot[b] = vals[rows].mean()
    return float(vals.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

ci_rows = []
for scope in ["overall"] + sorted(merged.snr_db.unique()):
    dscope = delta_df if scope == "overall" else delta_df[delta_df.snr_db == scope]
    for c in NONBASE:
        for metric, col in [("WER","delta_wer_pp"),("CER","delta_cer_pp")]:
            sub = dscope[dscope.condition == c]
            mean, lo, hi = cluster_ci(sub, col)
            ci_rows.append({"scope": scope, "condition": c, "metric": metric,
                            "n": len(sub), "mean_delta_pp": round(mean,4),
                            "ci_lo_pp": round(lo,4), "ci_hi_pp": round(hi,4)})
ci_df = pd.DataFrame(ci_rows)
print(ci_df.to_string(index=False))
ci_df.to_csv(DERIVED / "tier1_merged_bootstrap_ci.csv", index=False)

In [ ]:
def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    for rank, i in enumerate(order):
        adj[i] = p[i] * (m - rank)
    adj_sorted = np.minimum(1.0, np.maximum.accumulate(adj[order]))
    out = np.empty(m)
    out[order] = adj_sorted
    return np.minimum(1.0, out)

sig_rows = []
for metric, col in [("WER","delta_wer_pp"),("CER","delta_cer_pp")]:
    pvals = []
    for c in NONBASE:
        x = delta_df[delta_df.condition == c][col].to_numpy(float)
        w = stats.wilcoxon(x, alternative="two-sided", method="auto")
        dz = float(x.mean()/x.std(ddof=1)) if x.std(ddof=1)>0 else float("nan")
        pvals.append(float(w.pvalue))
        sig_rows.append({"metric": metric, "condition": c, "n": len(x),
                        "mean_delta_pp": round(float(x.mean()),4), "cohen_dz": round(dz,4),
                        "wilcoxon_statistic": float(w.statistic), "p_value": w.pvalue,
                        "win": int((x>0).sum()), "loss": int((x<0).sum()), "tie": int((x==0).sum())})
sig_df = pd.DataFrame(sig_rows)
for metric in ["WER","CER"]:
    mask = sig_df.metric == metric
    sig_df.loc[mask, "p_holm"] = holm_adjust(sig_df.loc[mask,"p_value"].to_numpy())
sig_df["p_value"] = sig_df["p_value"].round(6)
sig_df["p_holm"] = sig_df["p_holm"].round(6)
print(sig_df.to_string(index=False))
sig_df.to_csv(DERIVED / "tier1_merged_significance.csv", index=False)

## 3. Timing table from raw CSV

In [ ]:
r = raw.copy()
r["snr_db"] = r["snr_db"].fillna("clean").map(lambda v: "clean" if v == "clean" else str(int(float(v))))
r["noise_seed"] = r["noise_seed"].fillna("clean").astype(str)
r["merged_condition"] = r["condition"].map({"arm_a":"arm_a","hp":"high_pass","hp+sg":"high_pass",
                                          "hp_wiener":"high_pass","hp_ffmpeg":"hp_ffmpeg",
                                          "hp_wiener_oa":"hp_wiener_oa"})

timing_rows = []
for c in CONDITIONS:
    if c == "high_pass":
        sub = r[r.condition.isin(["hp","hp+sg","hp_wiener"])]
    else:
        sub = r[r.condition == c]
    timing_rows.append({"condition": c, "n": len(sub),
                       "rtf_mean": round(float(sub.rtf.mean()),4),
                       "rtf_sd": round(float(sub.rtf.std(ddof=1)),4),
                       "seconds_mean": round(float(sub.seconds.mean()),4),
                       "seconds_sd": round(float(sub.seconds.std(ddof=1)),4)})
timing_df = pd.DataFrame(timing_rows)

# paired overhead vs arm_a per scenario (high_pass = average of hp/hp+sg/hp_wiener)
keys = KEYS
arm_t = r[r.condition == "arm_a"].set_index(keys)
hp_t = r[r.condition.isin(["hp","hp+sg","hp_wiener"])].groupby(keys)[["rtf","seconds"]].mean()

over = {}
over["high_pass"] = (hp_t["rtf"] - arm_t["rtf"], hp_t["seconds"] - arm_t["seconds"])
for c in ["hp_ffmpeg","hp_wiener_oa"]:
    ct = r[r.condition == c].set_index(keys)
    over[c] = (ct["rtf"] - arm_t["rtf"], ct["seconds"] - arm_t["seconds"])

over_rows = []
for c in NONBASE:
    dr, ds = over[c]
    tmp = pd.DataFrame({"talk_id": [k[0] for k in dr.index], "delta_rtf": dr.to_numpy(), "delta_seconds": ds.to_numpy()})
    mr, lorr, hirr = cluster_ci(tmp, "delta_rtf")
    ms, lors, hiss = cluster_ci(tmp, "delta_seconds")
    over_rows.append({"condition": c,
                     "overhead_rtf_mean": round(mr,4), "overhead_rtf_ci_lo": round(lorr,4), "overhead_rtf_ci_hi": round(hirr,4),
                     "overhead_seconds_mean": round(ms,4), "overhead_seconds_ci_lo": round(lors,4), "overhead_seconds_ci_hi": round(hiss,4)})
over_df = pd.DataFrame(over_rows)
timing_df = timing_df.merge(over_df, on="condition", how="left")
timing_df.loc[timing_df.condition=="arm_a", ["overhead_rtf_mean","overhead_rtf_ci_lo","overhead_rtf_ci_hi",
                                            "overhead_seconds_mean","overhead_seconds_ci_lo","overhead_seconds_ci_hi"]] = 0.0
print(timing_df.to_string(index=False))
timing_df.to_csv(DERIVED / "tier1_merged_timing.csv", index=False)

## 4. Figures (four merged conditions only)

In [ ]:
m4 = merged[merged.condition.isin(CONDITIONS)].copy()
means = m4.groupby("condition")["wer"].mean().reindex(CONDITIONS)
fig, ax = plt.subplots(figsize=(7,5))
ax.bar(means.index, 100*means.values, color="#4c78a8")
ax.set_title("Tier 1 mean WER by merged condition")
ax.set_ylabel("WER (%)")
ax.set_xticks(range(len(means.index)))
ax.set_xticklabels(means.index, rotation=30, ha="right")
fig.tight_layout()
fig.savefig(DERIVED / "tier1_merged_wer_by_condition.png", dpi=150)
plt.close(fig)

snr_means = m4.groupby(["snr_db","condition"])["wer"].mean().unstack("condition").reindex(columns=CONDITIONS)
fig, ax = plt.subplots(figsize=(8,5))
snr_means.plot.bar(ax=ax)
ax.set_title("Tier 1 WER by input SNR and merged condition")
ax.set_ylabel("WER (%)")
fig.tight_layout()
fig.savefig(DERIVED / "tier1_merged_wer_by_snr.png", dpi=150)
fig.savefig(DERIVED / "tier1_merged_wer_by_bucket.png", dpi=150)
plt.close(fig)
print("Saved figures to", DERIVED)

## 5. Summary

In [ ]:
print("Saved tables:")
for f in ["tier1_merged_accuracy.csv","tier1_merged_paired_deltas.csv","tier1_merged_bootstrap_ci.csv",
          "tier1_merged_significance.csv","tier1_merged_timing.csv","tier1_merged_conditions.csv"]:
    print(" -", DERIVED / f)
print("Saved figures:")
for f in ["tier1_merged_wer_by_condition.png","tier1_merged_wer_by_snr.png","tier1_merged_wer_by_bucket.png"]:
    print(" -", DERIVED / f)

print()
for _, row in acc_df[acc_df.scope=="overall"].iterrows():
    print(f"  {row.condition}: WER {row.wer_mean_pct:.2f}%, CER {row.cer_mean_pct:.2f}% (n={row.n})")
print()
for _, row in ci_df[(ci_df.scope=='overall')&(ci_df.metric=='WER')].iterrows():
    print(f"  {row.condition} vs arm_a delta WER = {row.mean_delta_pp:+.3f} pp (95% CI {row.ci_lo_pp:+.3f}, {row.ci_hi_pp:+.3f})")
print()
print("Missing expected columns reported above.")